# Notebook 02: Web Scraping & Text Extraction

**Time:** 30 minutes  
**Prerequisites:** Notebook 01 complete  
**Goal:** Extract clean text from web pages using traditional and modern tools

This notebook will:
1. Extract text from web pages with trafilatura (traditional approach)
2. Use Crawl4AI for LLM-ready markdown extraction (modern 2025 approach)
3. Scrape arXiv paper abstracts for a pretraining dataset
4. Compare extraction quality between tools

> **Why this matters:** The quality of LLM pretraining data starts with extraction. Garbage in, garbage out. Modern tools like Crawl4AI (50K+ GitHub stars) produce cleaner, LLM-optimized markdown directly, while trafilatura remains a solid baseline for simpler extraction tasks.

In [1]:
import os, sys, time, importlib
from pathlib import Path

notebook_dir = os.getcwd()
parent_dir   = str(Path(notebook_dir).parent)
if parent_dir not in sys.path:
    sys.path.insert(0, parent_dir)

from dotenv import load_dotenv
load_dotenv(os.path.join(parent_dir, '.env'), override=True)

import src.llm_client, src.cost_tracker, src.utils, src.config
for mod in [src.llm_client, src.cost_tracker, src.utils, src.config]:
    importlib.reload(mod)

from src.llm_client import LLMClient
from src.cost_tracker import CostTracker
from src.utils import format_response, append_to_reflection
import src.config as config

# Week 3 specific imports
import src.scraping_utils
importlib.reload(src.scraping_utils)
from src.scraping_utils import (
    extract_with_trafilatura,
    scrape_arxiv_abstracts,
    compare_extractors,
)

client  = LLMClient(path=config.PATH)
tracker = CostTracker()

outputs_dir = os.path.join('..', 'outputs')
os.makedirs(outputs_dir, exist_ok=True)

print("Setup complete -- ready for Notebook 02")

✓ Ollama client initialized
  Available models: ['nomic-embed-text:latest', 'qwen3.5:latest', 'gemma4:e2b', 'llama3:latest', 'granite4:latest']
  Default model: llama3:latest
Setup complete -- ready for Notebook 02


---

## Part 1: Common Crawl & trafilatura

Common Crawl is a massive public archive of web pages -- like a snapshot of the entire internet, updated monthly. It's used to train models like GPT, Claude, and LLaMA.

**trafilatura** is a Python library that extracts the main content from web pages, stripping away navigation, ads, and boilerplate. Think of it as using a highlighter on millions of web pages to extract just the important sentences.

In [2]:
print("=" * 65)
print("Experiment 1: trafilatura Text Extraction")
print("=" * 65)
print()

# Extract text from an arXiv abstract page
url = "https://info.51.ca/articles/1533718"
result = extract_with_trafilatura(url)

print(f"\nExtracted text preview:")
print(result['text'][:500] if result['text'] else 'No text extracted')

Experiment 1: trafilatura Text Extraction

[trafilatura] Fetching: https://info.51.ca/articles/1533718
  Extracted 1,038 chars in 2.5s

Extracted text preview:
全国回暖，GTA却还在跌！多伦多房价年内或再降4.5%
一份最新报告显示，加拿大各地房地产市场有望很快回暖，但多伦多可能还会继续下跌。
根据 Royal LePage 发布的《房价调查与市场预测》，“2026 年第一季度，全国房屋综合均价同比下降 2%，但较 2025 年第四季度小幅上涨 0.7%。预计今夏同比将上涨 1%。”
不过，多伦多的情况并不乐观，今年第一季度均价下滑了 4.7%。Royal LePage 预计第二季度多伦多房价还将继续小幅下降。
报告指出，经济压力和地缘局势不稳定导致年初市场低迷。美国和以色列对伊朗的战争对经济产生了连锁反应，进而影响到房地产市场。
Royal LePage 总裁兼 CEO Phil Soper 在新闻稿中补充，市场疲软还受到了首次购房者观望、“先卖后买”行为，以及优质地段房源紧缺的影响。
GTA 年内或再降 4.5%。
Royal LePage 经纪 Shawn Zigelstein 指出，虽然多伦多房屋销量在 2025 年底同比略有上升，但价格基本持平，“高库存水平让市场保持平衡。”
他表示，首次购房者和“以小换大”的业主让 Condo 市


In [3]:
print("=" * 65)
print("Experiment 2: Extract from a Blog Post")
print("=" * 65)
print()

# Try a blog/news article
blog_url = "https://ai.meta.com/blog/meta-llama-4/"
blog_result = extract_with_trafilatura(blog_url)

print(f"\nExtracted {blog_result['char_count']:,} chars")
print(f"Preview: {blog_result['text'][:400]}..." if blog_result['text'] else 'No text extracted')

Experiment 2: Extract from a Blog Post

[trafilatura] Fetching: https://ai.meta.com/blog/meta-llama-4/
  Extracted 120 chars in 0.4s

Extracted 120 chars
Preview: You might find what you need on our homepage .
Foundational models
Our approach
Research
Meta AI
Latest news
Meta © 2026...


In [4]:
# TODO 1: Extract text from a URL of your choice
#
# Pick a web page related to your interests (blog post, docs page, wiki article).
# Extract the text and evaluate the quality.

# my_url = "[STUDENT: PASTE YOUR URL HERE]"
my_url = "https://news.popyard.space/cn16scroll19426400.html"

print("=" * 65)
print("TODO 1: Custom URL Extraction")
print("=" * 65)
print()

my_result = extract_with_trafilatura(my_url)
print(f"\nExtracted {my_result['char_count']:,} chars")
print(f"Preview: {my_result['text'][:1300]}..." if my_result['text'] else 'No text extracted')

todo1_reflection = """
[YOUR REFLECTION HERE]
This url is a news article about a celebrity's life story. It is a dynamic page with a lot of images and ads and lazy loading content
It did extract some content, but it missed a lot of the main text and included some irrelevant parts like ads and image captions. The extracted text was also somewhat jumbled and not well-structured.

"""

print()
print(todo1_reflection)

TODO 1: Custom URL Extraction

[trafilatura] Fetching: https://news.popyard.space/cn16scroll19426400.html
  Extracted 32,976 chars in 1.4s

Extracted 32,976 chars
Preview: 0
https://gears.popyard.space
https://www.popyard.space/img/logo_0.jpg
https://www.popyard.space
https://news.popyard.space
https://newsx.popyard.space
https://cn.popyard.space
https://plus.popyard.space
https://studio.popyard.space
https://studio.popyard.space
https://channel.popyard.space
https://adserver.popyard.space
八阕
新 闻
6
news-scroll
8次背叛，3次捉奸在床，她到底图什么？
16
1942640
1942640
0
5
0
与君説
2026-04-17
https://bbs.popyard.space
https://i.imgur.com/lSgOgfH.png
desktop
pixfuture
underjeff
vdoai
0
0
normal
{ "object_name":"related_list","lan_0":"cn","sub_0":"历经婚变、庇护所生活，单亲妈创业突围（组图）","sid_0":"16","rid_0":"1940351","gfw_0":"0","subject_0":"历经婚变、庇护所生活，单亲妈创业突围（组图）","lan_1":"cn","sub_1":"苍井空自曝和知名男星“一夜情”（组图）","sid_1":"16","rid_1":"1940434","gfw_1":"0","subject_1":"苍井空自曝和知名男星“一夜情”（组图）","lan_2":"cn","sub_2":"留子菜翻红，火成最新流量密码（组图）","s

---

## Part 2: Markdown Extraction -- Crawl4AI & html2text (2025)

The key insight of modern web scraping for LLMs: **output Markdown, not plain text**. Structured Markdown preserves headings, links, lists, and code blocks -- all of which help LLMs understand document structure.

**Crawl4AI** (50K+ GitHub stars) is the leading tool for this. It uses a headless browser to render JavaScript-heavy pages and outputs clean, LLM-ready Markdown. However, it requires **Python 3.10+**.

For Python 3.9 environments, we use **html2text** as a fallback -- it converts HTML to Markdown without a browser, which works great for static pages like arXiv.

| Tool | Output | JS Support | Python | Best For |
|------|--------|-----------|--------|----------|
| **trafilatura** | Plain text | No | 3.7+ | Bulk extraction, removing boilerplate |
| **html2text** | Markdown | No | 3.6+ | Static pages, lightweight Markdown |
| **Crawl4AI** | Markdown | Yes (browser) | 3.10+ | JS-heavy sites, LLM pipelines |

```
pip install html2text        # Lightweight fallback (works everywhere)
pip install crawl4ai         # Full-featured (requires Python 3.10+)
```

In [5]:
print("=" * 65)
print("Experiment 3: Markdown Extraction vs trafilatura")
print("=" * 65)
print()

# Compare both extractors on the same page
# Uses Crawl4AI if Python 3.10+, otherwise html2text as markdown fallback
comparison_url = "https://arxiv.org/abs/2404.00001"

try:
    comparison = compare_extractors(comparison_url)
    
    print("\n--- trafilatura output (plain text, first 300 chars) ---")
    print(comparison['trafilatura']['text'][:300])
    
    md_result = comparison['crawl4ai']
    print(f"\n--- {md_result.get('method', 'markdown')} output (first 300 chars) ---")
    print(md_result['text'][:300])
    
    print(f"\nKey difference: trafilatura gives plain text ({comparison['trafilatura']['char_count']:,} chars)")
    print(f"Markdown extractor preserves structure ({md_result['char_count']:,} chars)")
except Exception as e:
    print(f"Comparison failed: {e}")
    print("TIP: pip install html2text  (or pip install crawl4ai for Python 3.10+)")

Experiment 3: Markdown Extraction vs trafilatura

EXTRACTOR COMPARISON
URL: https://arxiv.org/abs/2404.00001

[trafilatura] Fetching: https://arxiv.org/abs/2404.00001
  Extracted 2,389 chars in 0.1s
[Crawl4AI] Fetching: https://arxiv.org/abs/2404.00001


[INIT].... → Crawl4AI 0.8.6 

[FETCH]... ↓ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.51s 

[SCRAPE].. ◆ https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.02s 

[COMPLETE] ● https://arxiv.org/abs/2404.00001                                                                     |
✓ | ⏱: 0.53s 

  Extracted 8,615 chars in 1.1s

--- Comparison ---
  trafilatura:  2,389 chars in 0.1s
  crawl4ai: 8,615 chars in 1.1s

--- trafilatura output (plain text, first 300 chars) ---
Physics > Physics Education
[Submitted on 3 Feb 2024]
Title:Uso de herramientas digitales matemáticas en la Educación Secundaria
View PDF HTML (experimental)Abstract:Information and Community Technologies (ICT) are very present in our society nowadays and particularly in the educative field. In just

--- crawl4ai output (first 300 chars) ---
[Skip to main content](https://arxiv.org/abs/2404.00001#content)
[![Cornell University](https://arxiv.org/static/browse/0.3.4/images/icons/cu/cornell-reduced-white-SMALL.svg)](https://www.cornell.edu/)
[Learn about arXiv becoming an independent nonprofit.](https://tech.cornell.edu/arxiv/)
We gratefu

Key difference: trafilatura gives plain text (2,389 chars)
Markdown extractor preserves structure (8,615 chars)


In [8]:
# TODO 2: Analyze the differences between extractors
#
# Use the LLM to analyze the quality differences between
# trafilatura and Crawl4AI outputs.

print("=" * 65)
print("TODO 2: LLM Analysis of Extraction Quality")
print("=" * 65)
print()

traf_text = blog_result['text'][:500] if blog_result['text'] else 'No extraction'

start = time.time()
response = client.generate(
    prompt=f"""Compare these two web extraction approaches for building LLM pretraining datasets:

1. **trafilatura** (traditional, 2020): Extracts plain text from HTML, removes boilerplate.
   Sample output: {traf_text[:200]}

2. **Crawl4AI** (modern, 2025): Produces LLM-ready Markdown with structure preserved.

For a team building a pretraining dataset:
- Which tool would you recommend for different scenarios?
- What are the trade-offs (speed, quality, features)?
- When would you use one over the other?""",
    system="You are an expert in data engineering for LLM pretraining.",
    max_tokens=400,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"Response in {elapsed:.1f}s")
    print(format_response(response, verbose=True))
else:
    print(f"Error: {response['error']}")

todo2_reflection = """
[YOUR REFLECTION HERE]
trafilatura trturns clean plain text without majoy issues for simple static page, 
but for javascript heavy dynamic pages it can miss some the main content.
Crawl4AI keeps some structure and can capture more content, 
but it may be slower and produce more verbose output.
For LLM usage, structured md type output can be more useful.

"""

print()
print(todo2_reflection)

TODO 2: LLM Analysis of Extraction Quality

Response in 87.0s
Model: llama3:latest
Tokens: 169 in, 349 out
Stop reason: complete
As an expert in data engineering for LLM pretraining, I'd be happy to help you decide which web extraction approach to use.

**Recommendations:**

1. **trafilatura**: Use this tool when:
	* You need a simple, straightforward text extraction process.
	* Your dataset requires minimal formatting and structure preservation.
	* Speed is crucial (trafilatura is generally faster).
2. **Crawl4AI**: Use this tool when:
	* You want to preserve the original structure and formatting of the web pages.
	* Your dataset requires more complex, LLM-ready Markdown output.
	* You're willing to trade off some speed for better quality.

**Trade-offs:**

1. **Speed:** trafilatura is generally faster than Crawl4AI due to its simpler text extraction approach.
2. **Quality:** Crawl4AI produces higher-quality output by preserving the original structure and formatting of web pages, whic

---

## Part 3: Building an arXiv Paper Dataset

Let's build a small pretraining dataset by scraping scientific paper abstracts from arXiv. This mirrors what real pretraining pipelines do at scale -- Meta's LLaMA 4 used academic papers as a key data source.

In [9]:
print("=" * 65)
print("Experiment 4: Scrape arXiv Papers")
print("=" * 65)
print()

papers = scrape_arxiv_abstracts(
    topic="large language models",
    max_results=5,
    save_path=os.path.join(outputs_dir, 'arxiv_papers.json'),
)

Experiment 4: Scrape arXiv Papers

Scraping arXiv: 'large language models' (max 5 papers)

  [1] Gravitational-wave lensing beyond rays: a disordered-system approach...
      Authors: Ripalta Amoruso, Ginevra Braga, Alice Garoffolo...
      Abstract: We develop a framework to describe gravitational wave propagation through a stochastic distribution of weak gravitationa...

  [2] LeapAlign: Post-Training Flow Matching Models at Any Generation Step by Building...
      Authors: Zhanhao Liang, Tao Yang, Jie Wu...
      Abstract: This paper focuses on the alignment of flow matching models with human preferences. A promising way is fine-tuning by di...

  [3] TokenLight: Precise Lighting Control in Images using Attribute Tokens...
      Authors: Sumit Chaturvedi, Yannick Hold-Geoffroy, Mengwei Ren...
      Abstract: This paper presents a method for image relighting that enables precise and continuous control over multiple illumination...

  [4] MM-WebAgent: A Hierarchical Multimodal Web Age

In [12]:
# TODO 3: Scrape papers on YOUR topic
#
# Choose a topic relevant to your capstone project or interests.
# Scrape 5-10 papers and save the results.

my_topic = "[AI safety]"  # e.g., "AI safety", "robotics", "NLP"

print("=" * 65)
print("TODO 3: Custom arXiv Scraping")
print("=" * 65)
print()

my_papers = scrape_arxiv_abstracts(
    topic=my_topic,
    max_results=8,
    save_path=os.path.join(outputs_dir, 'my_arxiv_papers.json'),
)

# Ask Claude to analyze the papers
paper_titles = [p['title'] for p in my_papers]
start = time.time()
response = client.generate(
    prompt=f"""I scraped these {len(my_papers)} papers from arXiv on the topic '{my_topic}':

{chr(10).join(f'{i+1}. {t}' for i, t in enumerate(paper_titles))}

Briefly analyze: What themes do you see? Which 2-3 papers seem most relevant for understanding current trends in this area?""",
    system="You are a research assistant helping analyze academic literature.",
    max_tokens=400,
    temperature=0.5
)
elapsed = time.time() - start

if "error" not in response:
    tracker.add_call(response)
    print(f"\nAnalysis ({elapsed:.1f}s):")
    print(format_response(response, verbose=False))

todo3_reflection = """
[YOUR REFLECTION HERE]
I picked AI safety and robotics, 
I wanted to see what recent papers are saying about the intersection of these fields and what the current research trends are. I was surprised that there were a lot of papers on using LLMs for robotics control and safety monitoring, 
which seems like a hot area right now. To scale this scraping approach for a real pretraining dataset, you would need to automate the process across many topics and sources, handle rate limits and CAPTCHAs, and implement robust error handling and data cleaning pipelines to ensure high-quality text extraction at scale.
I do this scraping approach can be a good way to build a focused pretraining dataset on specific research areas, but it would require significant engineering effort to scale and maintain over time. 
- What topic did you choose and why?
- Were you surprised by any of the papers found?
- How could this scraping approach scale to build a real pretraining dataset?
"""

print()
print(todo3_reflection)

TODO 3: Custom arXiv Scraping

Scraping arXiv: '[AI safety]' (max 8 papers)

  [1] RAD-2: Scaling Reinforcement Learning in a Generator-Discriminator Framework...
      Authors: Hao Gao, Shaoyu Chen, Yifan Zhu...
      Abstract: High-level autonomous driving requires motion planners capable of modeling multimodal future uncertainties while remaini...

  [2] Pure Borrow: Linear Haskell Meets Rust-Style Borrowing...
      Authors: Yusuke Matsushita, Hiromi Ishii
      Abstract: A promising approach to unifying functional and imperative programming paradigms is to localize mutation using linear or...

  [3] CoopEval: Benchmarking Cooperation-Sustaining Mechanisms and LLM Agents in Socia...
      Authors: Emanuel Tewolde, Xiao Zhang, David Guzman Piedrahita...
      Abstract: It is increasingly important that LLM agents interact effectively and safely with other goal-pursuing agents, yet, recen...

  [4] Simplifying Safety Proofs with Forward-Backward Reasoning and Prophecy...
      Author

---

## Summary & Reflection

In [13]:
_todo1 = todo1_reflection.strip() if 'todo1_reflection' in dir() else '[TODO 1 not completed yet]'
_todo2 = todo2_reflection.strip() if 'todo2_reflection' in dir() else '[TODO 2 not completed yet]'
_todo3 = todo3_reflection.strip() if 'todo3_reflection' in dir() else '[TODO 3 not completed yet]'

full_reflection = f"""
### Part 1 - trafilatura Extraction

{_todo1}

---

### Part 2 - Crawl4AI vs trafilatura

{_todo2}

---

### Part 3 - arXiv Paper Dataset

{_todo3}
"""

reflection_file = append_to_reflection(
    notebook="02",
    section_title="Web Scraping & Text Extraction",
    reflection_content=full_reflection,
    output_dir=os.path.join('..', 'outputs')
)

print(f"Reflection saved: {reflection_file}")
print()
tracker.report()

Reflection saved: ../outputs/homework_reflection.md

API COST REPORT
Total API calls:     5
Total input tokens:  996
Total output tokens: 1,733
Total cost:          $0.0000

Last 5 calls:
  1. [09:37:36] llama3:latest -- 169in/400out -- $0.0000
  2. [09:48:08] llama3:latest -- 169in/349out -- $0.0000
  3. [09:50:48] llama3:latest -- 218in/350out -- $0.0000
  4. [09:53:53] llama3:latest -- 220in/272out -- $0.0000
  5. [09:59:33] llama3:latest -- 220in/362out -- $0.0000


## Notebook 02 Complete!

**What you accomplished:**
- Extracted clean text from web pages with trafilatura
- Compared traditional vs modern (Crawl4AI) extraction approaches
- Built a mini paper dataset from arXiv

**Key concepts:**
- trafilatura removes HTML boilerplate to extract main content
- Crawl4AI produces LLM-ready Markdown with preserved structure
- arXiv API provides structured access to scientific papers

**Next:** Open **Notebook 03: Document OCR & PDF Extraction**